In [1]:
"""
6. Write a program for sentimental analysis (positive and negative) compare any 3 ML 
models. Do all the steps of preprocessing which you find relevant to task. 
"""

'\n6. Write a program for sentimental analysis (positive and negative) compare any 3 ML \nmodels. Do all the steps of preprocessing which you find relevant to task. \n'

In [2]:
import numpy as np
import pandas as pd
import re, sklearn

In [3]:
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

def lower(text):
    return text.lower()

def remove_punct(text):
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'https?://\S+', '', text)
    return text 

def tokenize(text):
    return word_tokenize(text)

def stop_words(tokens):
    stop_words = stopwords.words('english')
    return [ token for token in tokens if token not in stop_words and token.isalpha() ]

def stemming(tokens):
    stemmer = PorterStemmer()
    return [ stemmer.stem(token) for token in tokens ]

def preprocessing(text):
    text = lower(text)
    text = remove_punct(text)
    tokens = tokenize(text)
    tokens = stop_words(tokens)
    tokens = stemming(tokens)
    return ' '.join(tokens)

In [4]:
from nltk.corpus import movie_reviews

data = [ (movie_reviews.raw(fileid), movie_reviews.categories(fileid)[0]) for fileid in movie_reviews.fileids() ]
df = pd.DataFrame(data[:100] + data[-100:], columns=['text', 'category'])
df

,text,category
0,"plot : two teen couples go to a church party ,...",neg
1,the happy bastard's quick movie review \ndamn ...,neg
2,it is movies like these that make a jaded movi...,neg
3,""" quest for camelot "" is warner bros . ' firs...",neg
4,synopsis : a mentally unstable man undergoing ...,neg
...,...,...
195,wow ! what a movie . \nit's everything a movie...,pos
196,"richard gere can be a commanding actor , but h...",pos
197,"glory--starring matthew broderick , denzel was...",pos
198,steven spielberg's second epic film on world w...,pos


In [5]:
from sklearn.preprocessing import LabelEncoder

df['text'] = df['text'].apply(preprocessing)

encoder = LabelEncoder()
df['category'] = encoder.fit_transform(df['category'])
df

,text,category
0,plot two teen coupl go church parti drink driv...,0
1,happi bastard quick movi review damn bug got h...,0
2,movi like make jade movi viewer thank invent t...,0
3,quest camelot warner bro first featurelength f...,0
4,synopsi mental unstabl man undergo psychothera...,0
...,...,...
195,wow movi everyth movi funni dramat interest we...,1
196,richard gere command actor he alway great film...,1
197,glorystar matthew broderick denzel washington ...,1
198,steven spielberg second epic film world war ii...,1


In [6]:
X = df['text']
y = df['category']

from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(X).toarray()
vectorizer.get_feature_names_out()

array(['aaliyah', 'aaron', 'abandon', ..., 'zucker', 'zwick', 'zwigoff'],
      shape=(10599,), dtype=object)

In [7]:
test_data = data[100:125] + data[-125:-100]
test_df = pd.DataFrame(test_data, columns=['text', 'category'])

X_test = test_df['text'].apply(preprocessing)
X_test = vectorizer.transform(test_df['text']).toarray()
y_test = encoder.fit_transform(test_df['category'])

In [8]:
# Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

logistic = LogisticRegression()
logistic.fit(X, y)

y_pred = logistic.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.62      0.64      0.63        25
           1       0.62      0.60      0.61        25

    accuracy                           0.62        50
   macro avg       0.62      0.62      0.62        50
weighted avg       0.62      0.62      0.62        50



In [9]:
# Naive Bayes
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report

model = MultinomialNB()
model.fit(X, y)

y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.54      0.84      0.66        25
           1       0.64      0.28      0.39        25

    accuracy                           0.56        50
   macro avg       0.59      0.56      0.52        50
weighted avg       0.59      0.56      0.52        50



In [10]:
# SVM
from sklearn.svm import SVC
from sklearn.metrics import classification_report

svm = SVC()
svm.fit(X, y)

y_pred = svm.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.56      0.76      0.64        25
           1       0.62      0.40      0.49        25

    accuracy                           0.58        50
   macro avg       0.59      0.58      0.57        50
weighted avg       0.59      0.58      0.57        50



### Conclusion: It works better with Logistic Regression model